In [5]:
import pandas as pd
import numpy as np

# 1) Load data
input_file_path = "/home/ducvu0904/Downloads/xstk/injury_stat_enriched.csv"
output_file_path = "/home/ducvu0904/Downloads/xstk/Processed_dataset.csv"
df = pd.read_csv(input_file_path)

# 2) Standardize common missing markers
missing_markers = ["", " ", "NA", "N/A", "na", "null", "NULL", "-"]
df = df.replace(missing_markers, np.nan)

# Optional: normalize column names
df.columns = [c.strip() for c in df.columns]

# 3) Basic data quality checks
rows_before = len(df)
dup_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
rows_after = len(df)

print("=== DATA OVERVIEW ===")
print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Duplicate rows removed: {dup_count:,}")
print(f"Rows after duplicate removal: {rows_after:,}")
print(f"Columns: {df.shape[1]:,}")

# 4) Missing-value report for every feature (before fill)
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

nan_report = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": missing_count.values,
    "missing_pct": missing_pct.values,
    "n_unique": df.nunique(dropna=True).values
}).sort_values(["missing_count", "missing_pct"], ascending=False)

print("\n=== NaN REPORT (ALL FEATURES) - BEFORE FILL ===")
display(nan_report)

print("\n=== FEATURES WITH NaN > 0 - BEFORE FILL ===")
display(nan_report[nan_report["missing_count"] > 0])

# 5) Drop rows with NaN in required physical columns
required_cols_requested = ["player_height_inches", "player_weight"]
col_lookup = {c.lower(): c for c in df.columns}
missing_required = [c for c in required_cols_requested if c not in col_lookup]
if missing_required:
    raise KeyError(f"Required columns not found: {missing_required}")
required_cols = [col_lookup[c] for c in required_cols_requested]

rows_before_required_drop = len(df)
df = df.dropna(subset=required_cols).copy()
rows_dropped_required = rows_before_required_drop - len(df)
print("\n=== REQUIRED FIELDS CLEANING ===")
print(f"Rows dropped (NaN in {required_cols}): {rows_dropped_required:,}")
print(f"Rows remaining: {len(df):,}")

# 6) Fill NaN = 0.0 for all columns EXCEPT injury detail columns
protected_lower = {"injured_on", "returned", "days_missed", "injured_type"}
fill_cols = [c for c in df.columns if c.lower() not in protected_lower]
na_before_fill = int(df[fill_cols].isna().sum().sum())
df[fill_cols] = df[fill_cols].fillna(0.0)
na_after_fill = int(df[fill_cols].isna().sum().sum())

print("\n=== FILL MISSING VALUES ===")
print(f"Protected columns (kept NaN): {sorted(protected_lower)}")
print(f"Columns filled with 0.0: {len(fill_cols):,}")
print(f"NaN count in fill columns before: {na_before_fill:,}")
print(f"NaN count in fill columns after : {na_after_fill:,}")

# 7) NaN report after fill
missing_count_after = df.isna().sum()
missing_pct_after = (missing_count_after / len(df) * 100).round(2)
nan_report_after = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": missing_count_after.values,
    "missing_pct": missing_pct_after.values,
    "n_unique": df.nunique(dropna=True).values
}).sort_values(["missing_count", "missing_pct"], ascending=False)

print("\n=== NaN REPORT (ALL FEATURES) - AFTER FILL ===")
display(nan_report_after)

# Keep cleaned dataframe for next steps and save to final processed dataset
df_clean = df.copy()
df_clean.to_csv(output_file_path, index=False)
print(f"\nSaved processed data to: {output_file_path}")

=== DATA OVERVIEW ===
Rows before duplicate removal: 5,572
Duplicate rows removed: 0
Rows after duplicate removal: 5,572
Columns: 115

=== NaN REPORT (ALL FEATURES) - BEFORE FILL ===


,feature,dtype,missing_count,missing_pct,n_unique
28,INJURED_ON,str,4358,78.21,756
29,RETURNED,str,4358,78.21,736
30,DAYS_MISSED,float64,4358,78.21,64
31,INJURED_TYPE,str,4358,78.21,5
102,PULL_UP_FG_PCT,float64,352,6.32,377
...,...,...,...,...,...
107,SECONDARY_AST,float64,0,0.00,16
108,POTENTIAL_AST,float64,0,0.00,188
109,AST_ADJ,float64,0,0.00,124
110,AST_TO_PASS_PCT,float64,0,0.00,188



=== FEATURES WITH NaN > 0 - BEFORE FILL ===


,feature,dtype,missing_count,missing_pct,n_unique
28,INJURED_ON,str,4358,78.21,756
29,RETURNED,str,4358,78.21,736
30,DAYS_MISSED,float64,4358,78.21,64
31,INJURED_TYPE,str,4358,78.21,5
102,PULL_UP_FG_PCT,float64,352,6.32,377
104,PULL_UP_EFG_PCT,float64,352,6.32,464
112,OREB_CHANCE_PCT,float64,76,1.36,481
113,DREB_CHANCE_PCT,float64,76,1.36,412
114,REB_CHANCE_PCT_ADJ,float64,76,1.36,409
87,DIST_MILES_OFF,float64,1,0.02,18



=== REQUIRED FIELDS CLEANING ===
Rows dropped (NaN in ['PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT']): 0
Rows remaining: 5,572

=== FILL MISSING VALUES ===
Protected columns (kept NaN): ['days_missed', 'injured_on', 'injured_type', 'returned']
Columns filled with 0.0: 111
NaN count in fill columns before: 936
NaN count in fill columns after : 0

=== NaN REPORT (ALL FEATURES) - AFTER FILL ===


,feature,dtype,missing_count,missing_pct,n_unique
28,INJURED_ON,str,4358,78.21,756
29,RETURNED,str,4358,78.21,736
30,DAYS_MISSED,float64,4358,78.21,64
31,INJURED_TYPE,str,4358,78.21,5
0,PLAYER_ID,int64,0,0.00,1387
...,...,...,...,...,...
110,AST_TO_PASS_PCT,float64,0,0.00,188
111,AST_TO_PASS_PCT_ADJ,float64,0,0.00,214
112,OREB_CHANCE_PCT,float64,0,0.00,481
113,DREB_CHANCE_PCT,float64,0,0.00,412



Saved processed data to: /home/ducvu0904/Downloads/xstk/Processed_dataset.csv


In [4]:
dist_col = col_lookup.get("dist_miles_off", "DIST_MILES_OFF")

missing_dist_rows = df_clean[df_clean[dist_col].isna()]

print(f"Rows with missing {dist_col}: {len(missing_dist_rows)}")
print("Row index(es):", missing_dist_rows.index.tolist())

display(missing_dist_rows)

Rows with missing DIST_MILES_OFF: 1
Row index(es): [2991]


,PLAYER_ID,PLAYER_NAME,SEASON,SEASON_NUM,AGE,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,GP,MIN,USG_PCT,...,PASSES_MADE,PASSES_RECEIVED,SECONDARY_AST,POTENTIAL_AST,AST_ADJ,AST_TO_PASS_PCT,AST_TO_PASS_PCT_ADJ,OREB_CHANCE_PCT,DREB_CHANCE_PCT,REB_CHANCE_PCT_ADJ
2991,1627755,Tyler Ulis,18-19,18.5,23.0,70.0,160.0,1,0.8,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
